In [ ]:
#El objetivo de la practica es trabajar con el merge, sobre el excel de datos
#telefonia cargamos los primeros 3 meses (Enero-Marzo)
import pandas as pd

ruta_fichero = r"C:\PCAD\Datos Telefonia Separados Meses Comerciales URL.xlsx"

fras_enero = pd.read_excel(ruta_fichero,sheet_name="Facturación Enero 2020", header=2)
fras_febrero = pd.read_excel(ruta_fichero,sheet_name="Facturación Febrero 2020", header=0)
fras_marzo = pd.read_excel(ruta_fichero,sheet_name="Facturación Marzo 2020", header=1)
franjas = pd.read_excel(ruta_fichero,sheet_name="Franja_Edades", header=0)
#Concatenar los 3 meses 
fras_trimestre = pd.concat([fras_enero,fras_febrero,fras_marzo],ignore_index=True)
#Eliminar las filas en blanco
fras_trimestre = fras_trimestre.dropna(how='all')
#Vamos a calcular la edad + la decada
fras_trimestre['Dias'] = (pd.Timestamp('today') - fras_trimestre['Fecha Nacimiento']).dt.days
fras_trimestre['Edad'] = fras_trimestre['Dias'] // 365
#Vamos a calcular la decada
fras_trimestre['Decada'] = (fras_trimestre['Edad'].astype(str).str[0] + '0').astype('Int64')
#1. Combinar (merge) con el df de Franjas, solo se puede combinar de 2 en 2
#los dataframes, si el nombre del campo es el mismo usaremos on=
#si el nombre del campo es diferente, usaremos left_on (df izquierda) y right_on #para el dataframe de la derecha. el tipo de dato debe ser el mismo numero-numero, texto-texto, fecha-fecha.
franjas.info()
resultado = pd.merge(fras_trimestre, franjas, on="Decada", how='inner')
#2. Mostrar el dataframe, ojo el campo clave queda unico, es decir no pasa el cmpo de la tabla de la derecha. El how='inner' los que coinciden de los 2 dataframes.
resultado

#1. El merge con diferentes nombres
df_merged = pd.merge(df1, df2, left_on="ID_Cliente", right_on="Codigo_Cliente", how='inner')

#2. Combinar varios merge
df_merged1 = pd.merge(df1, df2, on="ID_Cliente", how='inner')
df_merged2 = pd.merge(df_merged1, df3, on="ID_Producto", how='inner')

#3. Optimizado concatenar merges (a partir del segundo merge, solo poner un df)
df_merged = pd.merge(df1, df2, on="ID_Cliente", how='inner').merge(df3, on="ID_Producto", how='inner')

#4. Solo traer un campo
#antes de combinar tienes que filtrar columnas
franjas = franjas[['Decada','Franja']]

In [ ]:
#Como se trabaja el how= en el merge
import pandas as pd

df_clientes = pd.DataFrame({
    "id":[1,2,3,4],
    "nombre":['Marta','Juan','Monica','Pedro']
})
#Vamos a comprobar si funciona
print(df_clientes.head(1))

df_pedidos = pd.DataFrame({
    "id_cliente":[1,2,2,5],
    "producto":['Teclado','Mouse','Monitor','Impresora']
})
#Vamos a comprobar si funciona
print(df_pedidos.head(1))

In [ ]:
#El INNER JOIN (intersección) solo los registros que coinciden en los 2 dataframes
pd.merge(df_clientes,df_pedidos,left_on="id",right_on="id_cliente",how='inner')

In [ ]:
#El LEFT JOIN (intersección) todos los registros de la izquierda y los que coincidan con los de la derecha
pd.merge(df_clientes,df_pedidos,left_on="id",right_on="id_cliente",how='left')

In [ ]:
#El RIGHT JOIN (intersección) todos los registros de la derecha y los que coincidan con los de la izquierda
pd.merge(df_clientes,df_pedidos,left_on="id",right_on="id_cliente",how='right')

In [ ]:
#El OUTER JOIN (intersección) todos los registros de los 2 dataframes
pd.merge(df_clientes,df_pedidos,left_on="id",right_on="id_cliente",how='outer')

Practica 5: Partiendo del fichero de excel, "Datos telefonia separado meses..."
Necesitamos averiguar la suma de importe factura por nombre de comercial y descripción de la satisfacción, de las facturas de enero a junio del 2020

Practica 5.1: Y ademas ver el promedio de importe factura por zona de localidad
(solo para valientes)

In [ ]:
#Practica 5
#1. Llamar a la libreria pandas
import pandas as pd
#Ajustar la ruta del archivo
ruta_fichero = r"C:\PCAD\Datos Telefonia Separados Meses Comerciales URL.xlsx"
#2. Descargar los dataframes de los mese (enero-junio) + Comerciales + Descricpión Satisfacción
fras_enero = pd.read_excel(ruta_fichero,sheet_name="Facturación Enero 2020", header=2)
fras_febrero = pd.read_excel(ruta_fichero,sheet_name="Facturación Febrero 2020", header=0)
fras_marzo = pd.read_excel(ruta_fichero,sheet_name="Facturación Marzo 2020", header=1)
fras_abril = pd.read_excel(ruta_fichero,sheet_name="Facturación Abril 2020", header=1)
fras_mayo = pd.read_excel(ruta_fichero,sheet_name="Facturación Mayo 2020", header=1)
fras_junio = pd.read_excel(ruta_fichero,sheet_name="Facturación Junio 2020", header=2)
#3. Concatenar los 6 meses
fras_semestre = pd.concat([fras_enero,fras_febrero,fras_marzo,fras_abril, fras_mayo, fras_junio],ignore_index=True)
#4. Cargar los 2 dataframes que faltan
comerciales = pd.read_excel(ruta_fichero,sheet_name="Comerciales", header=0)
descripciones = pd.read_excel(ruta_fichero,sheet_name="Satisfacción Cliente", header=0)

In [ ]:
#5. Comprobar los dataframes
#fras_semestre
#comerciales
#descripciones

#6. Combinar con merge anidado
resultado = pd.merge(fras_semestre,comerciales,on="IdComercial",how='inner').merge(descripciones, left_on="Satisfacción", right_on="Satisfacción Cliente", how='inner')

#7. Cambiar el nombre de una columna
resultado = resultado.rename(columns={"Nombre_y":"Comercial"})

#8. Realizar la agrupación en pantalla, con reset index una serie se convierte en un dataframe
resultado.groupby(["Comercial","Descripción_y"])['Importe factura'].sum().reset_index()

#9. Crear una referencia al dataframe + guardarlo en un fichero csv
agrupacion = resultado.groupby(["Comercial","Descripción_y"])['Importe factura'].sum().reset_index()
agrupacion.to_csv(r"C:\PCAD\agrupacion martes.csv",sep=",")

In [ ]:
#Practica 5.1: Y ademas ver el promedio de importe factura por zona de localidad
localidades = pd.read_excel(ruta_fichero,sheet_name="Localidad", header=0)
localidades

Como obtener la localidad solo con el metodo split (la clase str)
frase = "Esto es un curso de python"

In [ ]:
#En este caso dividimos por el espacio blanco, valor por defecto
#frase.split("/")
frase = "Esto es un curso de python"
lista = frase.split()
lista[5]

In [ ]:
#Dividir la columna por un caracter, quedarnos con la segunda columna, indice 1
#quitar los espacios en blanco
localidades['Localidad'] = localidades['Localidad'].str.split("#").str[1].str.strip()

In [ ]:
#Dejar solo 2 columnas
localidades = localidades[['Localidad','Zona']]
localidades
#Vamos a combinar con el dataframe de resultado
resultado = resultado.merge(localidades,on="Localidad",how='inner')
resultado

In [ ]:
resultado.groupby('Zona')['Importe factura'].mean().round(2)


Conectarnos con la libreria pdfplumber a un pdf y obtener sus tablas, que cargaremos en uno o varios dataframes
- Primero instalar en nuestro PC, pdfplumber
- >pip install pdfplumber

In [ ]:
#Cargar las tablas de un pdf
#0. Importar las librerias
import pdfplumber
import pandas as pd

#1. Ajustar la ruta del fichero
ruta_pdf = r"C:\PCAD\Franjas edades.pdf"

#2. Crear una lista para almacenar las tablas extraidas
tablas = []

#3. Crear un bucle para ir cargando las tablas, miraremos las diferentes tablas, de las diferentes paginas
with pdfplumber.open(ruta_pdf) as pdf:
    for page in pdf.pages:     #Es que me recorra todas las paginas del pdf
        #Extraer las tablas de la pagina que estoy recorriendo
        tablas_extraidas = page.extract_tables()
        #Vamos a recorrer la colección tablas_extraidas
        for tabla in tablas_extraidas:
            df = pd.DataFrame(tabla)
            tablas.append(df)

#Acceder al dataframe resultante
tablas[0]

#lo mas optimo, es clonar el dataframe en un objeto
franjas = tablas[0]

#Hacer la primera fila como encabezado
franjas = franjas.rename(columns=franjas.iloc[0])

#Eliminar la fila 0, con axis = 0 elimino filas, con axis=1 elimino columnas
franjas = franjas.drop([0],axis=0).reset_index(drop=True)
franjas


In [ ]:
franjas

#Practica 6:
Aprovechando el dataframe de semestre (enero-junio 2020), cargar las tablas del pdf "Tablas Telefonia.pdf", hacer merge con las 3 tablas que contiene el pdf, para obtener la dimension franja, la dimensión Descripción, promociones a aplicar, aplicar el descuento al importe factura según su zona.
- Primero aplicar el descuento por Zona (lo tenemos en la tabla "Promociones a aplicar")
- Guardar en un csv, la suma de importe por Franja.
- Guardar en un csv, el numero de facturas por descripción de la satisfacción.